In [33]:
"""
USED FOR BERT EMBEDDINGS ONLY

filter_target_words.py,  
======================
Step 1: Filter rows from each corpus that contain at least one target word,
then save a lean CSV per corpus (and per time period for arXiv).

Run this ONCE. The outputs feed every downstream step (BERT, Word2Vec, etc).

Usage:
    python filter_target_words.py

Outputs (in ./filtered/):Used for BERT embeddings only
    arxiv_1990_2011.csv
    arxiv_2012_2019.csv
    arxiv_2020_2023.csv
    pubmed.csv
    guardian.csv
    filter_summary.csv   ← row counts before/after for each corpus+word
"""

import os
import re
import pandas as pd

# ─────────────────────────────────────────
# CONFIG — adjust paths if needed
# ─────────────────────────────────────────

ARXIV_CSV    = "./cs_papers_api.csv"
PUBMED_CSV   = "./PubMed_200k_RCT_train.csv"
GUARDIAN_CSV = "./guardian_articles.csv"

ARXIV_TEXT_COL    = "text"          # will be created as title + abstract
ARXIV_YEAR_COL    = "year"
PUBMED_TEXT_COL   = "abstract_text"
GUARDIAN_TEXT_COL = "bodyContent"

ARXIV_PERIODS = {
    "1990_2011": (1990, 2011),
    "2012_2019": (2012, 2019),
    "2020_2023": (2020, 2023),
}

TARGET_WORDS = [
    "neural", "memory", "cell", "agent",
    "network", "node", "port", "cloud", "stream",
]


TARGET_WORDS_PUBMED = [
    "neural", "memory", "cell", "agent", "network", "node"
]

TARGET_WORDS_GUARDIAN = [
    "port", "cloud", "stream",
]


OUTPUT_DIR = "./filtered"


# ─────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────

def contains_any_target(text: str, words: list[str]) -> bool:
    """
    Returns True if the text contains at least one target word
    as a whole word (not a substring — 'network' won't match 'networking').
    """
    for word in words:
        if re.search(r'\b' + re.escape(word) + r'\b', str(text), re.IGNORECASE):
            return True
    return False


def which_words_present(text: str, words: list[str]) -> str:
    """Returns a comma-separated list of which target words appear in the text."""
    found = [w for w in words
             if re.search(r'\b' + re.escape(w) + r'\b', str(text), re.IGNORECASE)]
    return ",".join(found)


def filter_and_save(df: pd.DataFrame, text_col: str,
                    label: str, output_path: str, TARGET_WORDS) -> pd.DataFrame:
    """
    Filters df to rows where text_col contains at least one target word.
    Adds a 'matched_words' column so you can see which words triggered each row.
    Saves to output_path and prints a summary.
    """
    total = len(df)

    mask = df[text_col].fillna("").apply(
        lambda t: contains_any_target(t, TARGET_WORDS)
    )
    filtered = df[mask].copy()
    filtered["matched_words"] = filtered[text_col].fillna("").apply(
        lambda t: which_words_present(t, TARGET_WORDS)
    )

    filtered.to_csv(output_path, index=False)

    kept = len(filtered)
    print(f"  [{label}]  {total:>10,} rows → {kept:>8,} kept "
          f"({kept/total*100:.1f}%)  → {output_path}")
    return filtered


def word_counts(df: pd.DataFrame, text_col: str, label: str, TARGET_WORDS) -> list[dict]:
    """Returns per-word row counts for the summary CSV."""
    rows = []
    for word in TARGET_WORDS:
        count = df[text_col].fillna("").apply(
            lambda t: bool(re.search(r'\b' + re.escape(word) + r'\b',
                                     str(t), re.IGNORECASE))
        ).sum()
        rows.append({"corpus": label, "word": word, "row_count": int(count)})
    return rows


# ─────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    summary_rows = []

    # ── arXiv ──────────────────────────────
    # print("\n[arXiv]  Loading...")
    # arxiv = pd.read_csv(ARXIV_CSV)
    # arxiv["year_only"] = pd.to_datetime(arxiv[ARXIV_YEAR_COL]).dt.year

    # combine title + abstract into one text column
    # arxiv[ARXIV_TEXT_COL] = (
    #     arxiv.get("title", pd.Series("", index=arxiv.index)).fillna("") + " " +
    #     arxiv.get("abstract", pd.Series("", index=arxiv.index)).fillna("")
    # ).str.strip()

    # for period_label, (start, end) in ARXIV_PERIODS.items():
    #     period_df = arxiv[
    #         (arxiv["year_only"] >= start) & (arxiv["year_only"] <= end)
    #     ].copy()

    #     out_path = os.path.join(OUTPUT_DIR, f"arxiv_{period_label}_pubmed.csv")
    #     filtered = filter_and_save(period_df, ARXIV_TEXT_COL,
    #                                f"arxiv_{period_label}", out_path, TARGET_WORDS_PUBMED)
    #     summary_rows.extend(word_counts(filtered, ARXIV_TEXT_COL, f"arxiv_{period_label}",TARGET_WORDS_PUBMED))

    # for period_label, (start, end) in ARXIV_PERIODS.items():
    #     period_df = arxiv[
    #         (arxiv["year_only"] >= start) & (arxiv["year_only"] <= end)
    #     ].copy()

    #     out_path = os.path.join(OUTPUT_DIR, f"arxiv_{period_label}_guardian.csv")
    #     filtered = filter_and_save(period_df, ARXIV_TEXT_COL,
    #                                f"arxiv_{period_label}", out_path, TARGET_WORDS_GUARDIAN)
    #     summary_rows.extend(word_counts(filtered, ARXIV_TEXT_COL, f"arxiv_{period_label}",TARGET_WORDS_GUARDIAN))


    # ── PubMed ─────────────────────────────
    print("\n[PubMed]  Loading...")
    pubmed = pd.read_csv(PUBMED_CSV)
    out_path = os.path.join(OUTPUT_DIR, "pubmed.csv")
    filtered_pubmed = filter_and_save(pubmed, PUBMED_TEXT_COL, "pubmed", out_path, TARGET_WORDS_PUBMED)
    summary_rows.extend(word_counts(filtered_pubmed, PUBMED_TEXT_COL, "pubmed", TARGET_WORDS_PUBMED))

    # ── Guardian ───────────────────────────
    print("\n[Guardian]  Loading...")
    guardian = pd.read_csv(GUARDIAN_CSV)
    out_path = os.path.join(OUTPUT_DIR, "guardian.csv")
    filtered_guardian = filter_and_save(guardian, GUARDIAN_TEXT_COL,
                                        "guardian", out_path, TARGET_WORDS_GUARDIAN)
    summary_rows.extend(word_counts(filtered_guardian, GUARDIAN_TEXT_COL, "guardian",TARGET_WORDS_GUARDIAN))


In [34]:
main()


[PubMed]  Loading...
  [pubmed]   2,211,861 rows →   30,540 kept (1.4%)  → ./filtered/pubmed.csv

[Guardian]  Loading...
  [guardian]     149,839 rows →    6,596 kept (4.4%)  → ./filtered/guardian.csv
